# VMC2026 Track 2 — Baseline Pipeline (Kaggle)

QMOS (SpeechMOS) + EmoCat (emotion2vec) + EMOS/VAD (Gemini) → gộp `answer.txt`.

**Trước khi chạy:** Accelerator = **GPU T4**, Internet = **On**. Add Data: `nguyenthanhlim/emotional-speech-dataset-esd` (test). Secrets: `GEMINI_API_KEY` (cho EMOS/VAD).

Chạy ngay được: **QMOS + EmoCat** trên ESD. EMOS/VAD chờ data Track 2 (cần `metadata.csv`).

## 0. Config — SỬA Ở ĐÂY

In [ ]:
import os

WAV_DIR = '/kaggle/input/emotional-speech-dataset-esd'   # << SỬA khi có data Track 2
METADATA_CSV = None   # << ví dụ '/kaggle/input/<track2-data>/metadata.csv' (cho Gemini)
OUT_DIR = '/kaggle/working'

RUN_QMOS, RUN_EMOCAT = True, True
RUN_EMOS = RUN_VAD = METADATA_CSV is not None
EMOTIONS5 = ['angry', 'happy', 'neutral', 'sad', 'surprised']

def list_wavs(d):
    return sorted(f for f in os.listdir(d) if f.lower().endswith('.wav'))

print('WAV_DIR:', WAV_DIR)
print('Số wav:', len(list_wavs(WAV_DIR)) if os.path.isdir(WAV_DIR) else '(chưa thấy thư mục)')

## 1. Cài đặt

In [ ]:
!pip install -q speechmos funasr librosa soundfile pandas google-genai loguru tqdm

## 2. QMOS — SpeechMOS (UTMOS, không cần fairseq)

In [ ]:
def run_qmos(wav_dir):
    import torch, librosa
    predictor = torch.hub.load('tarepan/SpeechMOS:v1.2.0', 'utmos22_strong', trust_repo=True)
    scores = {}
    for w in list_wavs(wav_dir):
        wave, _ = librosa.load(os.path.join(wav_dir, w), sr=16000, mono=True)
        wave_t = torch.from_numpy(wave).unsqueeze(0)
        scores[w] = float(predictor(wave_t, sr=16000).mean().item())
    return scores

qmos_scores = run_qmos(WAV_DIR) if RUN_QMOS else {}
print('QMOS xong:', len(qmos_scores))
list(qmos_scores.items())[:3]

## 3. EmoCat — emotion2vec+ large
Đã sửa bug bản gốc + lọc 5 lớp + chuẩn hóa tổng = 1.

In [ ]:
def run_emocat(wav_dir):
    from funasr import AutoModel
    model = AutoModel(model='iic/emotion2vec_plus_large', hub='hf')
    results = {}
    for w in list_wavs(wav_dir):
        rec = model.generate(os.path.join(wav_dir, w), granularity='utterance', extract_embedding=False)
        probs = {e: 0.0 for e in EMOTIONS5}
        for lab, sc in zip(rec[0]['labels'], rec[0]['scores']):
            name = lab.split('/')[-1]
            if name in probs:
                probs[name] = float(sc)
        total = sum(probs.values())
        if total > 0:
            probs = {k: v / total for k, v in probs.items()}
        results[w] = probs
    return results

emocat_probs = run_emocat(WAV_DIR) if RUN_EMOCAT else {}
print('EmoCat xong:', len(emocat_probs))
list(emocat_probs.items())[:2]

## 4. EMOS & VAD — Gemini (chờ data Track 2)
Chỉ chạy khi có `METADATA_CSV` + `GEMINI_API_KEY`. Bỏ comment khi data về.
Đối chiếu tên cột output `Gemini_VAD.py` trước khi gộp.

In [ ]:
emos_scores, vad_scores = {}, {}
if RUN_EMOS or RUN_VAD:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
        print('Đã nạp GEMINI_API_KEY từ Secrets')
    except Exception as e:
        print('Chưa nạp được key:', e)
    # !git clone -q https://github.com/voicemos-challenge/vmc2026-baselines.git /kaggle/working/vmc2026-baselines
    # !cd /kaggle/working/vmc2026-baselines/track2/EMOS && python Gemini_EMOS.py --metadata-path $METADATA_CSV --base-path $WAV_DIR --output-file /kaggle/working/emos.csv --start-row 1 --end-row 50 --workers 4
    # !cd /kaggle/working/vmc2026-baselines/track2/VAD && python Gemini_VAD.py --metadata-path $METADATA_CSV --base-path $WAV_DIR --output-file /kaggle/working/vad.csv --start-row 1 --end-row 50 --workers 4
    import pandas as pd
    if os.path.exists('/kaggle/working/emos.csv'):
        d = pd.read_csv('/kaggle/working/emos.csv'); emos_scores = dict(zip(d['uttID'], d['emos']))
    if os.path.exists('/kaggle/working/vad.csv'):
        d = pd.read_csv('/kaggle/working/vad.csv')
        for _, r in d.iterrows():
            vad_scores[r['uttID']] = (r.get('valence'), r.get('arousal'), r.get('dominance'))
print('EMOS:', len(emos_scores), '| VAD:', len(vad_scores))

## 5. Gộp answer.txt (tự bỏ cột thiếu)

In [ ]:
def fmt_cat(p):
    return '|'.join(f'{e}:{p[e]:.6g}' for e in EMOTIONS5)

def build_answer(out_path):
    wavs = list_wavs(WAV_DIR)
    have_cat = RUN_EMOCAT and len(emocat_probs) > 0
    have_vad = RUN_VAD and len(vad_scores) > 0
    cols = ['wav', 'QMOS', 'EMOS']
    if have_cat: cols.append('CAT')
    if have_vad: cols += ['VAL', 'ARO', 'DOM']
    with open(out_path, 'w') as f:
        f.write(','.join(cols) + '\n')
        for w in wavs:
            row = [w, f"{qmos_scores.get(w, 3.0):.6g}", str(emos_scores.get(w, 3))]
            if have_cat: row.append(fmt_cat(emocat_probs.get(w, {e: 0.2 for e in EMOTIONS5})))
            if have_vad:
                v = vad_scores.get(w, (3, 3, 3)); row += [str(v[0]), str(v[1]), str(v[2])]
            f.write(','.join(row) + '\n')
    print(f'Ghi {len(wavs)} dòng → {out_path} | cột: {cols}')

answer_path = os.path.join(OUT_DIR, 'answer.txt')
build_answer(answer_path)
!head -3 {answer_path}

## 6. Validate + zip

In [ ]:
import csv
with open(answer_path) as f:
    rows = list(csv.reader(f))
header = rows[0]
assert header[0] == 'wav' and 'QMOS' in header and 'EMOS' in header, 'Header sai'
for i, r in enumerate(rows[1:], 2):
    assert len(r) == len(header), f'Dòng {i} sai số cột'
print(f'OK: {len(rows)-1} dòng, header = {header}')
!cd /kaggle/working && zip -j submission_track2.zip answer.txt && unzip -l submission_track2.zip